# Do checks and quality flags for extracted data
1. Load data
2. Convert to correct format (numeric for numerical columns)
3. Sanity checks

In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import geopy as gpy
import time
import itertools
import regex as re
from matplotlib import pyplot as plt
from src.data import *
from src.plot_functions import *
from src.post_process_functions import *
from src.geocoding import *
from src.hazard_def import *
from src.impact_def import *
from src.sanity_checks import *

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/lseverino/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to /Users/lseverino/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/lseverino/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
#load data (model)
filename_in = 'llm_response_impact_labelled_reports_test_multiprompt_continue_v050925_21rep_meta-llama_llama-4-scout-17b-16e-instruct'#"labelled_reports_impacts_all_v080925" 'llm_response_impact_labelled_reports_test_multiprompt_continue_v050925_21rep_meta-llama_llama-4-scout-17b-16e-instruct'
data_path = DATA_OUT_LLMS #DATA_LABELLED DATA_OUT_LLMS  (depending on whether we want to process the LLM output or the labelled data)

## Load data
response_df = pd.read_csv(data_path / (filename_in + ".csv"))


In [3]:
response_df

,impactSubtype,impactValue,impactValueMin,impactValueMax,impactValuePrecision,impactUnit,valueAnnotation,country,location,locationAnnotation,...,endDay,dateAnnotation,hazards,hazardsAnnotation,appealCode,country_kw,reportDate,reportLink,disasterType,nathaz_text
0,Affected People,326788.0,NaN,NaN,exact,people,"['People Affected:326,788 people']",['Pakistan'],"['Sindh', 'Balochistan', 'Khyber Pakhtunkhwa']",['Pakistan endured an exceptionally intense mo...,...,NaN,['Pakistan endured an exceptionally intense mo...,"['Flood', 'Convective storm']",['Pakistan endured an exceptionally intense mo...,MDRPK026,Pakistan,2025-03-28 00:00:00,https://go-api.ifrc.org/api/downloadfile/88815...,Flood,['DREF Final Report Pakistan Flood August 2024...
1,Injured People,584.0,NaN,NaN,exact,people,['The monsoon season caused306 fatalities and5...,['Pakistan'],"['Balochistan', 'Khyber Pakhtunkhwa', 'Sindh',...",['Pakistan endured an exceptionally intense mo...,...,NaN,['Pakistan endured an exceptionally intense mo...,"['Flood', 'Convective storm']",['Pakistan endured an exceptionally intense mo...,MDRPK026,Pakistan,2025-03-28 00:00:00,https://go-api.ifrc.org/api/downloadfile/88815...,Flood,['DREF Final Report Pakistan Flood August 2024...
2,Human Deaths,306.0,NaN,NaN,exact,people,['The monsoon season caused306 fatalities and5...,['Pakistan'],"['Balochistan', 'Khyber Pakhtunkhwa', 'Sindh',...",['The monsoon season caused306 fatalities and5...,...,NaN,"['The monsoon season started in June2024, resu...","['Flood', 'Convective storm']",['Pakistan endured an exceptionally intense mo...,MDRPK026,Pakistan,2025-03-28 00:00:00,https://go-api.ifrc.org/api/downloadfile/88815...,Flood,['DREF Final Report Pakistan Flood August 2024...
3,Displaced People,9500.0,NaN,NaN,exact,residents,"['Sindh experienced acute urban flooding, part...",['Pakistan'],"['Badin', 'Dadu', 'Jacobabad']","['Sindh experienced acute urban flooding, part...",...,NaN,['Pakistan endured an exceptionally intense mo...,['Flood'],['Pakistan endured an exceptionally intense mo...,MDRPK026,Pakistan,2025-03-28 00:00:00,https://go-api.ifrc.org/api/downloadfile/88815...,Flood,['DREF Final Report Pakistan Flood August 2024...
4,Homeless People,15000.0,NaN,NaN,exact,houses,"['In regions such as KP and Sindh, particularl...",['Pakistan'],"['Khyber Pakhtunkhwa', 'Sindh', 'Chitral', 'Ba...","['In regions such as KP and Sindh, particularl...",...,NaN,['Pakistan endured an exceptionally intense mo...,"['Flood', 'Convective storm']",['Pakistan endured an exceptionally intense mo...,MDRPK026,Pakistan,2025-03-28 00:00:00,https://go-api.ifrc.org/api/downloadfile/88815...,Flood,['DREF Final Report Pakistan Flood August 2024...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
311,Human Deaths,1.0,NaN,NaN,exact,person,['NADMA reported that the state of Johor suffe...,['Malaysia'],['Johor'],['NADMA reported that the state of Johor suffe...,...,NaN,['NADMA reported that the state of Johor suffe...,['Flood'],['Heavy rains that started in December2016 con...,MDRMY003,Malaysia,2017-11-21 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=17...,Flood,['DREF operation n° MDRMY003 Glide n° FL-2017-...
312,Affected People,329468.0,NaN,NaN,exact,people,['According to the Department of Social Welfar...,['Philippines'],"['Regions I', 'II', 'III', 'CALABARZON', 'V', ...",['According to the Department of Social Welfar...,...,17.0,"['On16 October2016, at2:30 AM, Typhoon Sarika ...",['Tropical storm'],"['On16 October2016, at2:30 AM, Typhoon Sarika ...",MDRPH021,Philippines,2017-05-31 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=16...,Cyclone,['DREF n° MDRPH021 GLIDE n° TC-2016-000108-PHL...
313,Residential Buildings,12777.0,NaN,NaN,exact,houses,"['According to government data,12,777 houses w...",['Philippines'],"['Regions I, II, III, CALABARZON, V and CAR']",['According to the Department of Social Welfar...,...,17.0,"['On16 October2016, at2:30 AM, Typhoon Sarika ...",['Tropical st

In [4]:
#get rid of nans
response_df = response_df.dropna(subset=["nathaz_text"]) if "nathaz_text" in response_df.columns else response_df

In [5]:
response_df

,impactSubtype,impactValue,impactValueMin,impactValueMax,impactValuePrecision,impactUnit,valueAnnotation,country,location,locationAnnotation,...,endDay,dateAnnotation,hazards,hazardsAnnotation,appealCode,country_kw,reportDate,reportLink,disasterType,nathaz_text
0,Affected People,326788.0,NaN,NaN,exact,people,"['People Affected:326,788 people']",['Pakistan'],"['Sindh', 'Balochistan', 'Khyber Pakhtunkhwa']",['Pakistan endured an exceptionally intense mo...,...,NaN,['Pakistan endured an exceptionally intense mo...,"['Flood', 'Convective storm']",['Pakistan endured an exceptionally intense mo...,MDRPK026,Pakistan,2025-03-28 00:00:00,https://go-api.ifrc.org/api/downloadfile/88815...,Flood,['DREF Final Report Pakistan Flood August 2024...
1,Injured People,584.0,NaN,NaN,exact,people,['The monsoon season caused306 fatalities and5...,['Pakistan'],"['Balochistan', 'Khyber Pakhtunkhwa', 'Sindh',...",['Pakistan endured an exceptionally intense mo...,...,NaN,['Pakistan endured an exceptionally intense mo...,"['Flood', 'Convective storm']",['Pakistan endured an exceptionally intense mo...,MDRPK026,Pakistan,2025-03-28 00:00:00,https://go-api.ifrc.org/api/downloadfile/88815...,Flood,['DREF Final Report Pakistan Flood August 2024...
2,Human Deaths,306.0,NaN,NaN,exact,people,['The monsoon season caused306 fatalities and5...,['Pakistan'],"['Balochistan', 'Khyber Pakhtunkhwa', 'Sindh',...",['The monsoon season caused306 fatalities and5...,...,NaN,"['The monsoon season started in June2024, resu...","['Flood', 'Convective storm']",['Pakistan endured an exceptionally intense mo...,MDRPK026,Pakistan,2025-03-28 00:00:00,https://go-api.ifrc.org/api/downloadfile/88815...,Flood,['DREF Final Report Pakistan Flood August 2024...
3,Displaced People,9500.0,NaN,NaN,exact,residents,"['Sindh experienced acute urban flooding, part...",['Pakistan'],"['Badin', 'Dadu', 'Jacobabad']","['Sindh experienced acute urban flooding, part...",...,NaN,['Pakistan endured an exceptionally intense mo...,['Flood'],['Pakistan endured an exceptionally intense mo...,MDRPK026,Pakistan,2025-03-28 00:00:00,https://go-api.ifrc.org/api/downloadfile/88815...,Flood,['DREF Final Report Pakistan Flood August 2024...
4,Homeless People,15000.0,NaN,NaN,exact,houses,"['In regions such as KP and Sindh, particularl...",['Pakistan'],"['Khyber Pakhtunkhwa', 'Sindh', 'Chitral', 'Ba...","['In regions such as KP and Sindh, particularl...",...,NaN,['Pakistan endured an exceptionally intense mo...,"['Flood', 'Convective storm']",['Pakistan endured an exceptionally intense mo...,MDRPK026,Pakistan,2025-03-28 00:00:00,https://go-api.ifrc.org/api/downloadfile/88815...,Flood,['DREF Final Report Pakistan Flood August 2024...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
311,Human Deaths,1.0,NaN,NaN,exact,person,['NADMA reported that the state of Johor suffe...,['Malaysia'],['Johor'],['NADMA reported that the state of Johor suffe...,...,NaN,['NADMA reported that the state of Johor suffe...,['Flood'],['Heavy rains that started in December2016 con...,MDRMY003,Malaysia,2017-11-21 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=17...,Flood,['DREF operation n° MDRMY003 Glide n° FL-2017-...
312,Affected People,329468.0,NaN,NaN,exact,people,['According to the Department of Social Welfar...,['Philippines'],"['Regions I', 'II', 'III', 'CALABARZON', 'V', ...",['According to the Department of Social Welfar...,...,17.0,"['On16 October2016, at2:30 AM, Typhoon Sarika ...",['Tropical storm'],"['On16 October2016, at2:30 AM, Typhoon Sarika ...",MDRPH021,Philippines,2017-05-31 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=16...,Cyclone,['DREF n° MDRPH021 GLIDE n° TC-2016-000108-PHL...
313,Residential Buildings,12777.0,NaN,NaN,exact,houses,"['According to government data,12,777 houses w...",['Philippines'],"['Regions I, II, III, CALABARZON, V and CAR']",['According to the Department of Social Welfar...,...,17.0,"['On16 October2016, at2:30 AM, Typhoon Sarika ...",['Tropical st

In [6]:
#convert numerical columns
#convert numerical columns
num_cols = ["impactValue", "impactValueMin", "impactValueMax","startYear", "startMonth", "startDay", "endYear", "endMonth", "endDay"]
list_cols = ["country","location", "hazards", "valueAnnotation", "locationAnnotation", "dateAnnotation", "hazardsAnnotation", "annotation"]
list_cols = [key for key in list_cols if key in response_df.columns]
response_df_proc = format_output(response_df, num_cols=num_cols, list_cols=list_cols)




In [7]:
#add iso3
response_df_proc["country_iso3"] = response_df_proc["country"].apply(country_name_to_iso3)
response_df_proc["country_iso3_kw"] = response_df_proc["country_kw"].apply(country_name_to_iso3) if "country_kw" in response_df_proc.columns else None

In [8]:
#format units
from spacy.lang.en import English
from spacy.lang.punctuation import TOKENIZER_PREFIXES, TOKENIZER_SUFFIXES, TOKENIZER_INFIXES
from spacy.lang.en import TOKENIZER_EXCEPTIONS
from spacy.tokenizer import Tokenizer
from spacy.util import compile_prefix_regex, compile_suffix_regex, compile_infix_regex

## Sanity checks
1. ValueInText: ImpactValue should be in the original text
2. MaxPop: ImpactValue with “people” unit should be smaller than country’s population
3. OrigSentence: Annotation sentence should be present in the original text
4. UnknownImpType: Inferred impactType must be in the allowed list
5. UnknownHaz: Inferred hazardType must be in allowed list
6. Location partially undefined. 


In [9]:
response_df_proc.impactUnit.value_counts()

impactUnit
people                           77
houses                           22
homes                            10
hectares                          8
households                        8
                                 ..
acres of crops                    1
% of crop-farming households      1
% of livestock farmers            1
head of livestock and poultry     1
% of homes                        1
Name: count, Length: 63, dtype: int64

In [10]:
response_df_exploded = explode_lists(response_df_proc)



/Users/lseverino/Documents/PhD/Projects/Como/como_project4/src/post_process_functions.py:171: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  repeat_counts = df[list_columns].applymap(len).max(axis=1)


In [11]:
response_df_exploded.hazards.value_counts()

hazards
Flood                       1008
Mass movement                302
Convective storm             181
Other storm                  128
Extreme cold temperature     121
Epidemic                     120
Conflict                     106
Tropical storm                66
Wildfire                      47
Drought                       18
Name: count, dtype: int64

In [12]:
#value in original text
response_df_proc = flag_value_in_text(response_df_proc)

In [13]:
response_df_proc.value_in_text.value_counts()

value_in_text
True     226
False      7
Name: count, dtype: int64

In [20]:
response_df_proc[response_df_proc["value_in_text"] == False][["impactValue", "impactUnit", "valueAnnotation", "nathaz_text"]]

,impactValue,impactUnit,valueAnnotation,nathaz_text
61,61000.00,people,"[Number of people being assisted:61,165 people...",['1 OPERATION UPDATE Mozambique | Drought Emer...
127,38400.00,people,"[7,684 households were evacuated (with over38,...",['DREF Final Report Rwanda - Floods and Landsl...
147,8500.00,hectares,[According to the Algerian Minister of Agricul...,['DREF operation Operation n° MDRTN010 Date of...
196,4.50,million CHF,[Federation-wide DREF amount initially allocat...,"['1 OPERATION UPDATE Barbados, Grenada, Jamaic..."
238,2.20,million hectors,[about1 million hectors of maize field affecte...,['Page 1 / 21 DREF Operation Zambia Drought 20...
243,2.04,million people,[an average of2.04 million people are facing f...,['Page 1 / 21 DREF Operation Zambia Drought 20...
315,4.60,CHF million,"[Furthermore, damage to agriculture sector amo...",['DREF n° MDRPH021 GLIDE n° TC-2016-000108-PHL...


In [16]:
#impacted people must be less than population
country_pop = pd.read_csv(DATA_PATH / ("API_SP.POP.TOTL_DS2_en_csv_v2_131993/"+"API_SP.POP.TOTL_DS2_en_csv_v2_131993.csv"),sep=',', header=2)
country_pop = country_pop.dropna(how="all",axis=1)

response_df_proc = pop_cntry_check(response_df_proc, country_pop)

 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023


In [17]:
response_df_proc.pop_cntry_check.value_counts()

pop_cntry_check
True    77
Name: count, dtype: int64

In [18]:
response_df_proc[response_df_proc["pop_cntry_check"] == False]

,impactSubtype,impactValue,impactValueMin,impactValueMax,impactValuePrecision,impactUnit,valueAnnotation,country,location,locationAnnotation,...,appealCode,country_kw,reportDate,reportLink,disasterType,nathaz_text,country_iso3,country_iso3_kw,value_in_text,pop_cntry_check


In [ ]:
# check if impact are in list
def flag_impactSubtype(extracted_data, impact_list):
    def check_imp(x):
        return x["impactSubtype"] not in impact_list
    extracted_data["unknown_impactSubtype"] = np.nan
    extracted_data["unknown_impactSubtype"] = extracted_data.apply(check_imp, axis=1)
    return extracted_data

impact_list = impactSubtype_list
response_df_proc = flag_impactSubtype(response_df_proc, impact_list)
response_df_proc.unknown_impactSubtype.value_counts()

unknown_impactSubtype
False    274
True       1
Name: count, dtype: int64

In [ ]:
# save
savename = "flaged_" + res_savename
response_df_proc.to_csv(DATA_OUT_LLMS + savename, index=False)